Create a chatbot, that will be composed from two of your previous tasks: from RAG module and from Agent/MCP module, however, extended with additional functionallity: ability to query infromation on natural disaster. In order to do that, create an MCP server, that will query [CSV file](https://www.kaggle.com/datasets/brsdincer/all-natural-disasters-19002021-eosdis) [﻿](https://www.kaggle.com/datasets/brsdincer/all-natural-disasters-19002021-eosdis)with Pandas and return user responses to their question from chat

Test coverage is mandatory.

You can replace that CSV with any other from [the list](https://www.kaggle.com/datasets?search=csv&page=2).﻿[](https://www.kaggle.com/datasets?search=csv&page=2)

**At least one evaluation metric is defined, evaluated, and demonstrated** (quantitative or qualitative measure used to assess the performance, quality, and safety of generative model outputs). You need to have at least small evaluation data set and for at least one criterion

In [3]:
%pip install openai -q
%pip install python-dotenv -q
%pip install langchain -q
%pip install langchain-openai -q
%pip install mcp[cli] -q
%pip install pandas -q
%pip install uv -q
%pip install pywintypes -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pywintypes (from versions: none)
ERROR: No matching distribution found for pywintypes


In [7]:
# src/disasters_server/server.py

import os
import pandas as pd
from mcp.server.fastmcp import FastMCP

# Crea la instancia del servidor MCP
mcp = FastMCP("natural_disasters")

# Ruta al CSV (puedes usar absoluta o relativa)

# Load files
#    Final-Practical-Task-chatbot\disasters-server\data\1900_2021_DISASTERS.xlsx - emdat data.csv
#    Final-Practical-Task-chatbot\disasters-server\data\1970-2021_DISASTERS.xlsx - emdat data.csv

CSV_PATH_1900_2021_DISASTERS = os.environ.get("DISASTERS_CSV_PATH", "..\\disasters-server\\data\\1900_2021_DISASTERS.xlsx - emdat data.csv")
print(f"Loading disasters data from: {CSV_PATH_1900_2021_DISASTERS}")

# Cargar el CSV una sola vez
df_disasters = pd.read_csv(CSV_PATH_1900_2021_DISASTERS)
print(f"Disasters data loaded: {df_disasters.shape[0]} rows, {df_disasters.shape[1]} columns")

CSV_PATH_1970_2021_DISASTERS = os.environ.get("DISASTERS_CSV_PATH", "..\\disasters-server\\data\\1970_2021_DISASTERS.xlsx - emdat data.csv")
print(f"Loading disasters data from: {CSV_PATH_1970_2021_DISASTERS}")

df_disasters_1970_2021 = pd.read_csv(CSV_PATH_1970_2021_DISASTERS)
print(f"Disasters data loaded: {df_disasters_1970_2021.shape[0]} rows, {df_disasters_1970_2021.shape[1]} columns")

# unificar los df_disasters y df_disasters_1970_2021
df_disasters = pd.concat([df_disasters, df_disasters_1970_2021], ignore_index=True)
print(f"Combined disasters data: {df_disasters.shape[0]} rows, {df_disasters.shape[1]} columns")

Loading disasters data from: ..\disasters-server\data\1900_2021_DISASTERS.xlsx - emdat data.csv
Disasters data loaded: 16126 rows, 45 columns
Loading disasters data from: ..\disasters-server\data\1970_2021_DISASTERS.xlsx - emdat data.csv
Disasters data loaded: 14644 rows, 47 columns
Combined disasters data: 30770 rows, 47 columns


In [1]:
import sys
import json
sys.path.insert(0, "..\\disasters-server\\src")

from disasters_server.server import query_disasters

result = await query_disasters(country="Colombia", year=2021, limit=3)

parsed = json.loads(result)
print(json.dumps(parsed, indent=2, ensure_ascii=False, default=str))

Disasters data loaded: 14644 rows, 47 columns
Combined disasters data: 30770 rows, 47 columns
{
  "total": 3,
  "disasters": [
    {
      "Year": 2021,
      "Seq": 182,
      "Disaster Group": "Natural",
      "Disaster Subgroup": "Hydrological",
      "Disaster Type": "Flood",
      "Country": "Colombia",
      "ISO": "COL",
      "Region": "South America",
      "Continent": "Americas",
      "Location": "Florencia City (Caquetá Department); Quípama Town (Boyacá Department), Bogotá",
      "Origin": "Heavy rains",
      "Associated Dis": "Slide (land, mud, snow, rock)",
      "Dis Mag Scale": "Km2",
      "Start Year": 2021,
      "Start Month": 4.0,
      "Start Day": 1.0,
      "End Year": 2021,
      "End Month": 4.0,
      "End Day": 5.0,
      "Total Deaths": 3.0,
      "No Injured": 5.0,
      "No Affected": 360.0,
      "Total Affected": 365.0,
      "Adm Level": "2",
      "Admin2 Code": "13608;13691;13914",
      "Geo Locations": "Florencia, Quipama, Santafe De Bogota D.c.

In [ ]:
from dotenv import load_dotenv
from openai import AzureOpenAI
import os

def llm_client(message:str):
    load_dotenv()

    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    MODEL = os.getenv("MODEL")
    AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT")

    client = AzureOpenAI(
        api_key         = OPENAI_API_KEY,
        api_version     = "2024-08-01-preview",
        azure_endpoint  =  AZURE_ENDPOINT
    )

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": message
        }
    ]

    response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    )

    return response.choices[0].message.content

def get_prompt_to_identify_tool_and_arguments(query, tools): 
    tools_description = "\n".join([f"- {tool.name}, {tool.description}, {tool.inputSchema} " for tool in tools])
    return  ("You are a helpful assistant with access to these tools:\n\n"
                f"{tools_description}\n"
                "Choose the appropriate tool based on the user's question. \n"
                f"User's Question: {query}\n"                
                "If no tool is needed, reply directly.\n\n"
                "IMPORTANT: When you need to use a tool, you must ONLY respond with "                
                "the exact JSON object format below, nothing else:\n"
                "Keep the values in str "
                "{\n"
                '    "tool": "tool-name",\n'
                '    "arguments": {\n'
                '        "argument-name": "value"\n'
                "    }\n"
                "}\n\n")



[05/10/26 06:14:15] INFO     HTTP Request: POST                                                     ]8;id=13492395;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=13492396;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://ramiro-bedoya-3347-resource.openai.azure.com/openai/deployment                
                             s/gpt-5-nano/chat/completions?api-version=2024-08-01-preview "HTTP/1.1                
                             200 OK"                                                                               

The Beatles were a British rock band formed in Liverpool in 1960. The classic lineup was four members who played together for most of their career:

- John Lennon (born October 9, 1940 – died December 8, 1980)
  - Role: rhythm guitarist and lead vocalist; primary songwriter in partnership with Paul McCartney.
  - Notable for: a distinctive vocal style and songs like “Imagine,” “Strawberry Fields Forever,” “Help!” and “Come Together.” He was also a key figure in the band’s early evolution and later pursued solo projects with Yoko Ono.

- Paul McCartney (born June 18, 1942)
  - Role: bass guitarist and vocalist; major songwriter alongside Lennon.
  - Notable for: catchy melodies and wide-ranging work, including “Yesterday,” “Hey Jude,” “Let It Be,” and later Wings. Known for his melodic bass lines and multi-instrumental talents.

- George Harrison (born February 25, 1943 – died November 29, 2001)
  - Role: lead guitarist and occasional vocalist; contributed many songs and Indian music in